<a href="https://colab.research.google.com/github/catastrxphic/AI_Practice/blob/ImageGenerator_StableDiffusion/ImageGenerator_stableDiffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playing Around With Stable Diffusion's Code

### Aim:
###### Get a better understanding of what happens behind the curtains when creating images from text with AI

## Initializing Stable Diffusion

In [ ]:
''' installations needed:
- lpips (learned perceptual image patch similarity) -> judge simmilarity between images
- accelerate -> speed up training & improve GPU use
- diffusers -> generate images from text
'''
!pip install transformers diffusers lpips accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Imports & use:

# DS for tensors and deep learning model development functionalities
import torch
# encode text into embeddings and coverst text to numbers the model understands
from transformers import CLIPTextModel, CLIPTokenizer
# encode and decode images, denoising network in diffusion process, manage noise scheduling on diffusion steps
from diffusers import AutoencoderKL, UNet2DConditionModel, LMSDiscreteScheduler
# display progress bars
from tqdm.auto import tqdm
# enables faster computations with lower memory use
from torch import autocast
from matplotlib import pyplot as plt
# image manipulation
from PIL import Image
import numpy as np
# image transformations fro preprocessing
from torchvision import transforms as tfms

# video display
from IPython.display import HTML
from base64 import b64encode

# setting device:
torch_device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# load autoencoder to help decode latents into image space
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae", use_auth_token=True)

# load tokenizer & text encoder to encode and tokenize text
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# load UNET model to generate latents
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet", use_auth_token=True)

# load noise scheduler
scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)

# go to GPU
vae = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet = unet.to(torch_device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

## Inference

In [ ]:
# import drive module
from google.colab import drive
# bridging google drive and colab env
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Variables for image generation
prompt = [""]
height = 512
width = 768
inference_steps = 50
guidance_scale = 7.5
generator = torch.manual_seed(4)
batch_size = 1

In [ ]:
# prepate text
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
# torch no grad disables gradient calculation temporarily
with torch.no_grad():
  # encode and save text as encoded numbers
  text_embedding = text_encoder(text_input.input_ids.to(torch_device))[0]
max_len = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size, padding="max_length", max_length=max_len, return_tensors="pt"
)

with torch.no_grad():
  uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
text_embedding = torch.cat([uncond_embeddings, text_embedding])

In [ ]:
# # preparing scheduler
# scheduler.set_timesteps(inference_steps)

# # preparing latents
# latents = torch.randn(
#     (batch_size, unet.in_channels, height // 8, width // 8),
#     generator=generator,
# )

In [ ]:
# latents = latents.to(torch_device)
# latents = latents * scheduler.sigmas[0] # Need to scale to match k

#     # Loop
# with autocast("cuda"):
#     for i, t in tqdm(enumerate(scheduler.timesteps)):
#         # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
#         latent_model_input = torch.cat([latents] * 2)
#         sigma = scheduler.sigmas[i]
#         latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5)

#         # predict the noise residual
#         with torch.no_grad():
#             noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embedding)["sample"]

#         # perform guidance
#         noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
#         noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

#         # compute the previous noisy sample x_t -> x_t-1
#         latents = scheduler.step(noise_pred, i, latents)["prev_sample"]

# # scale and decode the image latents with vae
# latents = 1 / 0.18215 * latents

# with torch.no_grad():
#     image = vae.decode(latents)

# # Display
# image = (image / 2 + 0.5).clamp(0, 1)
# image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
# images = (image * 255).round().astype("uint8")
# pil_images = [Image.fromarray(image) for image in images]
# pil_images[0]

In [ ]:
# preparing scheduler
scheduler.set_timesteps(inference_steps)

# preparing latents
latents = torch.randn(
    (batch_size, unet.in_channels, height // 8, width // 8),
    generator=generator,
)

latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma # changed sigmas[0] to init_noise_sigma

# Loop
with autocast("cuda"):
    for i, t in tqdm(enumerate(scheduler.timesteps)):
        # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t) #moved out of loop

        # predict the noise residual
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embedding)["sample"]

        # perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # compute the previous noisy sample x_t -> x_t-1
        latents = scheduler.step(noise_pred, t, latents)["prev_sample"] # changed i to t

<ipython-input-11-e32320fc6f57>:6: FutureWarning: Accessing config attribute `in_channels` directly via 'UNet2DConditionModel' object attribute is deprecated. Please access 'in_channels' over 'UNet2DConditionModel's config object instead, e.g. 'unet.config.in_channels'.
  (batch_size, unet.in_channels, height // 8, width // 8),
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


0it [00:00, ?it/s]

In [ ]:
# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents

with torch.no_grad():
    image = vae.decode(latents)

# Display
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

NameError: name 'latents' is not defined